In [5]:
import numpy as np
import pandas as pd

from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score

# ============================================
# 0. Paths / basic settings
# ============================================
GA_NPZ_PATH = "../data/25_ga_best.npz"

N_TOP        = 5     # top-5 GA solutions
N_ITER_LIGHT = 20    # light tuning
RANDOM_STATE = 42

# Load train / test data
X_train = pd.read_csv("../data/X_train_xgbsel.csv")
X_test  = pd.read_csv("../data/X_test_xgbsel.csv")

y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/y_test.csv").squeeze()

# Convert labels to 1D numpy vectors
y_tr = y_train.to_numpy().ravel()
y_te = y_test.to_numpy().ravel()

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

# ============================================
# 2. Load GA results (npz)
# ============================================
ga_npz = np.load(GA_NPZ_PATH, allow_pickle=True)
print("\nnpz keys:", ga_npz.files)

# Make sure these names match the actual keys in the npz
masks  = ga_npz["top_masks"]   # shape: [n_solutions, n_features]
scores = ga_npz["top_scores"]  # GA internal scores (assume larger is better)

print("masks shape:", masks.shape)
print("scores shape:", scores.shape)

# ============================================
# 3. Get top-5 indices by GA score
# ============================================
top_idx = np.argsort(scores)[::-1][:N_TOP]

print("\nTop-5 GA solutions (by GA score):")
for r, idx in enumerate(top_idx, 1):
    print(
        f"  Rank {r}: idx={idx}, GA_score={scores[idx]:.4f}, "
        f"n_features={masks[idx].sum()}"
    )

# ============================================
# 4. Hyperparameter search space (for light tuning)
# ============================================
PARAM_DIST = {
    "n_estimators": randint(100, 600),
    "max_depth": [None, 4, 8, 12, 16],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": randint(2, 20),
    "min_samples_leaf": randint(1, 8),
    "bootstrap": [True],
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=2042)
results = []

# ============================================
# 4.5 Baseline: use all columns (84 features)
# ============================================
print("\n" + "=" * 60)
print("[Baseline] ALL 84 features (no GA mask)")

Xtr_all = X_train.to_numpy()   # use all features
Xte_all = X_test.to_numpy()

rs_bl = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=2042, n_jobs=-1),
    param_distributions=PARAM_DIST,
    n_iter=N_ITER_LIGHT,
    scoring="roc_auc",
    cv=cv5,
    n_jobs=-1,
    random_state=2042,
    verbose=0,
)
rs_bl.fit(Xtr_all, y_tr)

bl_params = dict(rs_bl.best_params_)
bl_params.pop("random_state", None)
bl_params.pop("n_jobs", None)
if bl_params.get("bootstrap") is False:
    bl_params.pop("max_samples", None)

print(f"[Baseline CV] mean AUC = {rs_bl.best_score_:.4f}")
print(f"[Baseline CV] best params = {bl_params}")

rf_bl = RandomForestClassifier(
    random_state=RANDOM_STATE, n_jobs=-1, **bl_params
)
rf_bl.fit(Xtr_all, y_tr)
y_bl = rf_bl.predict_proba(Xte_all)[:, 1]
auc_bl = roc_auc_score(y_te, y_bl)

print(f"[Baseline TEST] AUC = {auc_bl:.4f}")

results.append(
    {
        "rank": 0,
        "ga_idx": -1,
        "ga_score": np.nan,
        "n_features": X_train.shape[1],
        "cv_auc": float(rs_bl.best_score_),
        "test_auc": float(auc_bl),
        "params": bl_params,
    }
)

# ============================================
# 5. For each top-5 feature set:
#    - light RandomizedSearchCV
#    - retrain on full train subset
#    - compute test AUC
# ============================================
for rank, idx in enumerate(top_idx, 1):
    mask = masks[idx].astype(bool)
    feat_idx = np.where(mask)[0]
    n_feat = len(feat_idx)

    print("\n" + "=" * 60)
    print(
        f"[Rank {rank}] GA idx={idx}, GA_score={scores[idx]:.4f}, "
        f"n_features={n_feat}"
    )
    print("Feature indices:", feat_idx)

    Xtr_sub = X_train.iloc[:, feat_idx].to_numpy()
    Xte_sub = X_test.iloc[:, feat_idx].to_numpy()

    # --- Light RandomizedSearchCV ---
    rs = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=2042, n_jobs=-1),
        param_distributions=PARAM_DIST,
        n_iter=N_ITER_LIGHT,
        scoring="roc_auc",
        cv=cv5,
        n_jobs=-1,
        random_state=2042,
        verbose=0,
    )
    rs.fit(Xtr_sub, y_tr)

    best_params = dict(rs.best_params_)
    # Remove random_state and n_jobs before passing to RF directly
    best_params.pop("random_state", None)
    best_params.pop("n_jobs", None)
    if best_params.get("bootstrap") is False:
        best_params.pop("max_samples", None)

    print(f"[CV] mean AUC = {rs.best_score_:.4f}")
    print(f"[CV] best params = {best_params}")

    # --- Train on full train subset + evaluate on TEST ---
    rf = RandomForestClassifier(
        random_state=RANDOM_STATE, n_jobs=-1, **best_params
    )
    rf.fit(Xtr_sub, y_tr)
    y_pred = rf.predict_proba(Xte_sub)[:, 1]
    auc_te = roc_auc_score(y_te, y_pred)

    print(f"[TEST] AUC (top-{rank}) = {auc_te:.4f}")

    results.append(
        {
            "rank": rank,
            "ga_idx": int(idx),
            "ga_score": float(scores[idx]),
            "n_features": int(n_feat),
            "cv_auc": float(rs.best_score_),
            "test_auc": float(auc_te),
            "params": best_params,
        }
    )

# ============================================
# 6. Print summary table
# ============================================
results_df = pd.DataFrame(results)
print("\n===== SUMMARY: Baseline + Top-5 GA subsets (light tuning) =====")
print(results_df[["rank", "ga_idx", "ga_score", "n_features", "cv_auc", "test_auc"]])


X_train: (82, 84) X_test: (36, 84)
y_train: (82,) y_test: (36,)

npz keys: ['run_ids', 'top_scores', 'top_masks', 'top_cols']
masks shape: (5, 84)
scores shape: (5,)

Top-5 GA solutions (by GA score):
  Rank 1: idx=0, GA_score=0.8434, n_features=25
  Rank 2: idx=1, GA_score=0.8137, n_features=24
  Rank 3: idx=2, GA_score=0.8137, n_features=24
  Rank 4: idx=3, GA_score=0.7919, n_features=25
  Rank 5: idx=4, GA_score=0.7876, n_features=27

[Baseline] ALL 84 features (no GA mask)
[Baseline CV] mean AUC = 0.8024
[Baseline CV] best params = {'bootstrap': True, 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 4, 'n_estimators': 446}
[Baseline TEST] AUC = 0.7535

[Rank 1] GA idx=0, GA_score=0.8434, n_features=25
Feature indices: [ 2  3  4  5  6 12 13 17 21 25 29 31 34 35 36 40 41 53 59 63 64 66 71 81
 83]
[CV] mean AUC = 0.8327
[CV] best params = {'bootstrap': True, 'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 5, 